# Visualización de nódulos procesados

Carga los parches CT y máscaras generados por `process_all_masks.py` y permite explorarlos visualmente.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [ ]:
# Cargar metadata
output_dir = "output"
meta = pd.read_csv(os.path.join(output_dir, "metadata.csv"))
print(f"Total nódulos: {len(meta)}")
meta.head(10)

In [ ]:
# Distribución de malignancy
plt.figure(figsize=(8, 4))
plt.hist(meta["malignancy"], bins=20, edgecolor="black")
plt.xlabel("Malignancy (media radiólogos)")
plt.ylabel("Nº nódulos")
plt.title("Distribución de malignancy")
plt.show()

In [ ]:
# Elegir un nódulo por índice de la tabla
nodule_row = 0  # <-- cambia este valor para explorar otros nódulos

row = meta.iloc[nodule_row]
fname = f"{row.patient_id}_nod{int(row.nodule_idx)}.npy"

ct = np.load(os.path.join(output_dir, "CT", fname))
mask = np.load(os.path.join(output_dir, "masks", fname))

print(f"Paciente: {row.patient_id}")
print(f"Nódulo: {int(row.nodule_idx)}")
print(f"Shape: {ct.shape}")
print(f"Anotaciones: {int(row.num_annotations)}")
print(f"Malignancy: {row.malignancy}")

In [ ]:
# Visualizar el slice central: CT, máscara, y overlay
slice_idx = ct.shape[2] // 2

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(ct[:, :, slice_idx], cmap="gray")
axes[0].set_title("CT")
axes[0].axis("off")

axes[1].imshow(mask[:, :, slice_idx], cmap="gray")
axes[1].set_title("Máscara")
axes[1].axis("off")

axes[2].imshow(ct[:, :, slice_idx], cmap="gray")
axes[2].imshow(mask[:, :, slice_idx], alpha=0.4, cmap="Reds")
axes[2].set_title("CT + Máscara")
axes[2].axis("off")

plt.suptitle(f"{row.patient_id} — nódulo {int(row.nodule_idx)} — malignancy {row.malignancy}", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Visualizar todos los slices del nódulo
n_slices = ct.shape[2]
cols = min(n_slices, 6)
rows_grid = (n_slices + cols - 1) // cols

fig, axes = plt.subplots(rows_grid, cols, figsize=(3 * cols, 3 * rows_grid))
axes = np.atleast_2d(axes)

for i in range(rows_grid * cols):
    r, c = divmod(i, cols)
    ax = axes[r, c]
    if i < n_slices:
        ax.imshow(ct[:, :, i], cmap="gray")
        ax.imshow(mask[:, :, i], alpha=0.4, cmap="Reds")
        ax.set_title(f"z={i}", fontsize=9)
    ax.axis("off")

plt.suptitle(f"{row.patient_id} — nódulo {int(row.nodule_idx)} — todos los slices", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Comparar varios nódulos lado a lado (slice central de cada uno)
n_show = min(8, len(meta))
fig, axes = plt.subplots(2, n_show, figsize=(3 * n_show, 6))

for i in range(n_show):
    r = meta.iloc[i]
    f = f"{r.patient_id}_nod{int(r.nodule_idx)}.npy"
    ct_i = np.load(os.path.join(output_dir, "CT", f))
    mask_i = np.load(os.path.join(output_dir, "masks", f))
    z = ct_i.shape[2] // 2

    axes[0, i].imshow(ct_i[:, :, z], cmap="gray")
    axes[0, i].set_title(f"mal={r.malignancy}", fontsize=9)
    axes[0, i].axis("off")

    axes[1, i].imshow(ct_i[:, :, z], cmap="gray")
    axes[1, i].imshow(mask_i[:, :, z], alpha=0.4, cmap="Reds")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("CT", fontsize=11)
axes[1, 0].set_ylabel("CT + Máscara", fontsize=11)
plt.suptitle("Comparación de nódulos (slice central)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Resumen de features por nódulo
features = ["subtlety", "sphericity", "margin", "lobulation", "spiculation", "texture", "malignancy"]
meta[features].describe().round(2)